# 04 — Modelos, calibração e níveis (dias 7–10)

O treino "oficial" é feito por linha de comando (reprodutível):
```bash
python -m src.train --dataset v1 --versao v1
python -m src.train --dataset v1 --versao v1-metas --limiares metas --nao-salvar
```
Aqui analisamos os resultados e fazemos experimentos.

In [ ]:
# Preparação: permite importar src/ a partir da pasta notebooks/
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 50)
print("Raiz do projeto:", RAIZ)

In [ ]:
import json
VERSAO = "v1"   # "sintetico" para testar
metricas = json.loads((config.DIR_RELATORIOS / f"metricas_{VERSAO}.json").read_text())
print((config.DIR_RELATORIOS / f"comparacao_{VERSAO}.md").read_text())

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(config.DIR_RELATORIOS / f"curva_pr_{VERSAO}.png")))
display(Image(filename=str(config.DIR_RELATORIOS / f"calibracao_{VERSAO}.png")))

In [ ]:
pd.read_csv(config.DIR_RELATORIOS / f"importancia_features_{VERSAO}.csv")

## Níveis: fixos × metas
Quantas horas de teste cairiam em cada nível? Com limiares fixos, ALTO/CRITICO quase nunca aparecem? (recomendação R3)

In [ ]:
from src.artefatos import carregar_pacote
from src.dataset import carregar
from src.niveis import niveis_por_probabilidade

pacote = carregar_pacote(VERSAO)
dataset, _ = carregar(pacote.metadados["dataset"])
teste = dataset[dataset.split == "teste"]
p = pacote.prever_probabilidade(teste)
for nome, lim in {"fixos": config.LIMIARES_NIVEIS_PADRAO, "modelo": pacote.limiares}.items():
    niv = niveis_por_probabilidade(p, lim)
    print(nome, lim, np.bincount(niv, minlength=4))
print("maior probabilidade no teste:", p.max().round(3), "| p99,9:", np.quantile(p, 0.999).round(4))

## Estudo de caso: um dia com alagamento na Zona Norte

In [ ]:
zn = teste.assign(p=p)[teste.zona == "NORTE"]
dia_evento = zn.loc[zn.alagou_3h == 1, "data_hora_utc"].dt.tz_convert("America/Sao_Paulo").dt.date.value_counts().index[0]
caso = zn[zn.data_hora_utc.dt.tz_convert("America/Sao_Paulo").dt.date == dia_evento]
ponto = caso.loc[caso.alagou_3h == 1, "ponto_id"].iloc[0]
serie = caso[caso.ponto_id == ponto].set_index("data_hora_utc")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(serie.index, serie.chuva_1h, width=0.03, label="chuva 1h (mm)")
ax2 = ax.twinx()
ax2.plot(serie.index, serie.p, color="red", label="probabilidade")
ax2.scatter(serie.index[serie.alagou_3h == 1], serie.p[serie.alagou_3h == 1], color="black", zorder=3, label="alagou em 3h")
ax.set_title(f"{ponto} em {dia_evento}"); fig.legend(loc="upper left")

## Experimento R1 — chuva prevista
Só com dataset de previsões históricas: `python -m src.dataset --versao v1prev --fonte-chuva previsoes_historicas` (depois de baixar a chuva com `--fonte previsoes_historicas`) e `python -m src.train --dataset v1prev --versao v1prev --usar-chuva-prevista --nao-salvar`.